# Capability — PPE Detection
Locate protective equipment on an assembly line and highlight every instance with normalized Perceptron geometry.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericpence/perceptron_repo/blob/main/cookbook/recipes/capabilities/object-detection/object-detection.ipynb)

## Install dependencies
Install the SDK plus Pillow so we can preview grounded overlays.

In [ ]:
!uv pip install --upgrade perceptron pillow

## Configure the Perceptron client
Authenticate once, then resolve the PPE asset for the remaining cells.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw

from perceptron import configure, image, perceive, text
from perceptron.pointing.geometry import scale_box_to_pixels

PERCEPTRON_API_KEY = os.environ.get("PERCEPTRON_API_KEY", "<your Perceptron API key>")
MODEL_NAME = "isaac-0.1"

configure(
    provider="perceptron",
    api_key=PERCEPTRON_API_KEY,
)

BASE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/detection/"
SCENE_PATH = Path("ppe_line.webp")
ANNOTATED_PATH = Path("ppe_line_annotated.png")
if not SCENE_PATH.exists():
    urlretrieve(BASE_URL + SCENE_PATH.name, SCENE_PATH)

print(f"Calling Perceptron | input={SCENE_PATH}")


## Build a detection helper
Use the `@perceive` decorator with `expects="box"` so each detection returns normalized bounding boxes.

In [ ]:
TARGET_CLASSES = ["safety helmet", "safety vest"]


@perceive(model=MODEL_NAME, expects="box", allow_multiple=True)
def detect_ppe(frame_path):
    frame = image(frame_path)
    classes_text = ", ".join(TARGET_CLASSES)
    prompt = text(
        "Find every worker wearing PPE. Focus on helmets and high-visibility vests. "
        "Return one bounding box per instance and include the item name in the mention attribute."
    )
    return frame + prompt

> Tip: Reach for the high-level `detect()` helper when you need a quick, single-call detector. Use `@perceive` when you want to customize prompts, stream results, or change the expected geometry. Swap `expects="box"` for `"point"` / `"polygon"` when you need different shapes.


## Run the detection request
Invoke the helper on the PPE line image to retrieve grounded regions.

In [ ]:
detection = detect_ppe(str(SCENE_PATH))
print(detection.text)
boxes = detection.points or []
print(f"Returned {len(boxes)} boxes")

## Render grounded results
Convert the normalized coordinates to pixels and overlay them for quick inspection.

In [ ]:
img = Image.open(SCENE_PATH).convert("RGB")
draw = ImageDraw.Draw(img)

for box in boxes:
    scaled = scale_box_to_pixels(box, width=img.width, height=img.height)
    top_left = scaled.top_left
    bottom_right = scaled.bottom_right
    tlx, tly = int(round(top_left.x)), int(round(top_left.y))
    brx, bry = int(round(bottom_right.x)), int(round(bottom_right.y))
    draw.rectangle([tlx, tly, brx, bry], outline="dodgerblue", width=3)
    draw.text((tlx, tly), box.mention or "ppe", fill="dodgerblue")

img.save(ANNOTATED_PATH)
display(IPyImage(filename=str(ANNOTATED_PATH)))
print(f"Saved annotated output to {ANNOTATED_PATH}")


## Conclusion & next steps
- Adjust `TARGET_CLASSES` and the prompt to fit your environment.
- Enable `stream=True` inside `@perceive` for incremental detections.
- Add exemplar shots (see the in-context learning recipe) when classes are ambiguous.